In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 120)
sns.set_style('whitegrid')

print('pandas', pd.__version__)
print('numpy', np.__version__)
print('matplotlib', plt.__version__ if hasattr(plt,'__version__') else 'unknown')
print('seaborn', sns.__version__)

In [ ]:
file_path = r'C:\Users\HP\OneDrive\Documents\fifa\fifa_world_cup_2026_player_performance_cleaned.csv'
df = pd.read_csv(file_path)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
print(df.shape)

In [ ]:
df_cleaned = df.dropna(axis=1, how='any')

print(df_cleaned)


In [ ]:
# List of columns to drop (using your exact snake_case names)
cols_drop = [
    'yellow_cards', 'red_cards', 'offsides', 'saves', 'punches', 
    'save_percentage', 'clean_sheet', 'goals_conceded', 'penalty_saves',
    'jersey_number', 'age', 'height_cm', 'weight_kg', 'preferred_foot', 
    'club_name', 'market_value_eur', 'match_id', 'stadium', 'city', 
    'player_rating', 'performance_score', 'offensive_contribution', 
    'defensive_contribution', 'possession_impact', 'pressure_resistance', 
    'creativity_score', 'consistency_score', 'clutch_performance_score', 
    'player_of_match_awards', 'tournament_rating'
]

cols_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_drop)

print(f"Columns remaining: {len(df.columns)}")

In [ ]:
df.head()

In [ ]:
# Fix missing columns if they aren't explicitly there
for column in ['minutes_played', 'goals', 'assists', 'shots']:
    if column not in df.columns:
        df[column] = 0

# Calculate advanced stats safely using your exact lowercase names
df['GoalsPer90'] = np.where(df['minutes_played'] > 0, df['goals'] / (df['minutes_played'] / 90.0), 0.0)
df['AssistsPer90'] = np.where(df['minutes_played'] > 0, df['assists'] / (df['minutes_played'] / 90.0), 0.0)
df['ShotConversion'] = np.where(df['shots'] > 0, df['goals'] / df['shots'], 0.0)

# Display the top rows using the correct column names from your data
df[['player_name', 'team', 'goals', 'GoalsPer90', 'AssistsPer90']].head()

In [ ]:
df.head()

In [ ]:
# Calculate top 10 scorers using correct lowercase names
player_goals = df.groupby('player_name')['goals'].sum().reset_index()
top_10_players = player_goals.nlargest(10, 'goals')

# Draw the chart
sns.barplot(
    x='goals', 
    y='player_name', 
    data=top_10_players, 
    hue='player_name', 
    palette='Blues_r', 
    legend=False
)

# Add the goal numbers on the bars
for bar in plt.gca().containers:
    plt.gca().bar_label(bar, fmt='%d', padding=5)

# Set simple titles
plt.title('Top 10 Highest Goal Scorers', fontsize=14, pad=15)
plt.xlabel('Total Goals')
plt.ylabel('Player')

plt.show()

In [ ]:
country_goals = df.groupby('team')['goals'].sum().reset_index()
country_goals = country_goals.sort_values(by='goals', ascending=False)

# Grab the top 7 countries and group the rest as 'Others'
top_countries = country_goals.head(7)
others_goals = country_goals.iloc[7:]['goals'].sum()

# pie chart
pie_data = pd.concat([top_countries, pd.DataFrame([{'team': 'Others', 'goals': others_goals}])])
pie_data = pie_data[pie_data['goals'] > 0]

plt.figure(figsize=(8, 8))
plt.pie(pie_data['goals'], labels=pie_data['team'], autopct='%1.1f%%', startangle=140)

plt.title('Country-Wise Distribution of Total Goals', fontsize=14, pad=20)
plt.show()

In [ ]:
# Total high-level dataset metrics
total_goals = int(df['goals'].sum())
total_players = df['player_name'].nunique()
total_teams = df['team'].nunique()

top_scorer = df.groupby('player_name')['total_goals_tournament'].max()
best_scorer = top_scorer.idxmax()
most_goals = int(top_scorer.max())

# 3. Print the high-level metrics clearly
print("===== FIFA WORLD CUP 2026 SUMMARY =====")
print(f"Total Matches Sampled : {len(df):,}")
print(f"Total Unique Players  : {total_players}")
print(f"Total Teams / Nations : {total_teams}")
print(f"Total Match Goals     : {total_goals}")
print(f"\nTop Scorer Overall    : {best_scorer} ({most_goals} goals)")

position_summary = df.groupby('position').agg(
    Players=('player_name', 'nunique'),
    GoalsScored=('goals', 'sum')
)

print("\n===== POSITION SUMMARY =====")
print(position_summary)